# EOM Experiment Code

This is simple IQ loopback test program.

## QICK Pyro4 Instanctiation

In [12]:
import numpy as np
import matplotlib.pyplot as plt
import time
from qcodes import (
    Parameter,
    Measurement,
    Station,
    load_or_create_experiment,
    initialise_or_create_database_at,
)

from qick import *
from qick.averager_program import QickSweep
from qick.pyro import make_proxy

# Qick version : 0.2.357
(soc, soccfg) = make_proxy("192.168.2.99")

# Set DAC Channel 0 attenuation 20 dB and 20 dB, and turn on DAC channel
soc.rfb_set_gen_rf(0,0,0)
# Set DAC Channel filter as bypass mode
soc.rfb_set_gen_filter(0,fc = 2.5, ftype = "lowpass")

# Set ADC Channel attenuation 20 dB, and turn on ADC channel
soc.rfb_set_ro_rf(0,0)
# Set ADC Channel filter as bypass mode
soc.rfb_set_ro_filter(0, fc = 2.5, ftype = "lowpass")

Pyro.NameServer PYRO:Pyro.NameServer@0.0.0.0:8888
myqick PYRO:obj_e5ec5821c09246e686cd55834fd8256b@192.168.2.99:37475


# QCodes Setup

In [11]:
station = Station()

initialise_or_create_database_at("C:/Users/Measurement6/Nextcloud2/Lab/Data/QSim/2025/20251127_QICK_Test/test.db")
exp = load_or_create_experiment("2D sweep", "test_data")
meas = Measurement(exp=exp, station=station)

meas_I = Parameter(name = "I", label = "I", unit = "V")
meas_Q = Parameter(name = "Q", label = "Q", unit = "V")
meas.register_parameter(meas_I)
meas.register_parameter(meas_Q)

## Program

In [15]:
class EOM_Pulse_Test(NDAveragerProgram):
    def initialize(self):
        self.phrst = 0
        freq_rf    = 200
        # Declare RF generation channel
        self.declare_gen(
            ch      = 0,        # Channel
            nqz     = 1         # Nyquist Zone
        )
        # Declare DC generation channel
        self.declare_gen(
            ch      = 5,        # Channel
            nqz     = 1         # Nyquist Zone
        )
        # Declare RF input channel
        self.declare_readout(
            ch      = 0,        # Channel
            length  = self.cfg["pulse_length"]  # Readout length
        )

        # Convert RF frequency to DAC DDS register value
        freq_dac    = self.freq2reg(
            f       = freq_rf,  # Frequency
            gen_ch  = 0,        # Generator channel
            ro_ch   = 0         # Readout channel for round up
        )
        # Convert RF frequency to ADC DDS register value
        freq_adc    = self.freq2reg_adc(
            f       = freq_rf,  # Frequency
            ro_ch   = 0,        # Readout channel
            gen_ch  = 0         # Generator channel for round up
        )

        # Set DAC DDS
        self.set_pulse_registers(
            ch      = 0,        # Generator channel
            style   = "const",  # Output is gain * DDS output
            freq    = freq_dac, # Generator DDS frequency
            phase   = 45,        # Generator DDS phase
            gain    = 5000,     # Generator amplitude
            length  = self.cfg["pulse_length"],     # Pulse length
            phrst   = self.phrst# Generator DDS phase reset
        )
        
        # Set DC
        self.set_pulse_registers(
            ch      = 5,        # Generator channel
            style   = "const",  # Output is gain * DDS output
            freq    = 0,        # Generator DDS frequency
            phase   = 0,        # Generator DDS phase
            gain    = 32000,     # Generator amplitude
            length  = self.cfg["pulse_length"],     # Pulse length
            phrst   = self.phrst# Generator DDS phase reset
        )
        # Set ADC DDS
        self.set_readout_registers(
            ch      = 0,        # Readout channel
            freq    = freq_adc, # Readout DDS frequency
            length  = self.cfg["pulse_length"],      # Readout DDS multiplication length
            phrst   = self.phrst# Readout DDS phase reset
        )
        self.synci(500)

    def body(self):
        self.pulse(
            ch      = 0,        # Generator channel
            t       = 50        # Pulse will be output @ sync_t + 100
        )
        self.pulse(
            ch      = 5,        # Generator channel
            t       = 50        # Pulse will be output @ sync_t + 100
        )
        self.readout(
            ch      = 0,        # Readout channel
            t       = 50        # Readout DDS will start multiplication
                                # @ sync_t + 100
        )
        self.trigger(
            adcs    = [0],      # Readout channels
            adc_trig_offset = 150 # Readout will capture the data @ sync_t + 50
        )
        self.sync_all(5000)

## Execution

In [19]:
prog = EOM_Pulse_Test(
    soccfg,
    {
        "reps" : 1000,
        "expts" : 1,
        "pulse_length" : 1000,
    }
)
start_time = time.time()
expt_pts, avg_di, avg_dq = prog.acquire(soc, progress=True, start_src = "internal")
end_time = time.time()

total_time = end_time - start_time
print(prog)
print(f"total time : {total_time} s")


  0%|          | 0/1000 [00:00<?, ?it/s]


// Program

          regwi 0, $22, 178956971;              //freq = 178956971
          regwi 0, $23, 45;                     //phase = 45
          regwi 0, $25, 5000;                   //gain = 5000
          regwi 0, $26, 590824;                 //phrst| stdysel | mode | | outsel = 0b01001 | length = 1000 
          regwi 2, $12, 0;                      //freq = 0
          regwi 2, $13, 0;                      //phase = 0
          regwi 2, $15, 32000;                  //gain = 32000
          regwi 2, $16, 17368040;               //phrst| stdysel | mode | | outsel = 0b100001001 | length = 1000 
          regwi 4, $22, 357913942;              //freq = 357913942
          regwi 4, $26, 1000;                   //mode | outsel = 0b00000 | length = 1000 
          synci 500;
          regwi 0, $13, 0;
          regwi 0, $14, 999;
LOOP_rep: regwi 0, $27, 50;                     //t = 50
          set 0, 0, $22, $23, $0, $25, $26, $27;//ch = 0, pulse @t = $27
          regwi 2, $17, 50